# Evaluation
Both the raw predictions and the ground truth are lists of true and false values.

To evaluate the model performances, we compare each model's predictions to the ground truth to derive confusion matrices and performance scores.

## Custom Modules
The ``src`` directory houses custom modules with functions that will be reused throughout the project.

To be able to import these moduls, we begin with programmatically adding the project's root directory to ``sys.path``.

After adding the root to ``sys.path``, we can import the ``data`` and ``util`` modules:

In [1]:
import os, sys

# recursively search for the root directory containing a specific file
def find_root_dir(search_for='.gitignore'):

    current_dir = os.getcwd()

    while True:
        if os.path.exists(os.path.join(current_dir, search_for)):
            return current_dir
        parent_dir = os.path.dirname(current_dir)
        if parent_dir == current_dir:
            raise FileNotFoundError(f"Could not find '{search_for}' in any parent directory.")
        current_dir = parent_dir


# save the root directory to a variable
root_dir = find_root_dir()
print(f"Root directory found: {root_dir}")

# add the root directory to the system path
sys.path.append(root_dir)
if root_dir in sys.path:
    print(f"Root directory added to system path.")


# import custom modules
from src import data

Root directory found: c:\dev\automated_title_abstract_screening
Root directory added to system path.


## Loading The Data
Begin by loading the data.

The predictions by the supervised machine learning models and the llama models are initially stored within different files.

In [2]:
path_data = '../../data/datasets/04_preprocessed'
path_predictions = '../../data/predictions'

datasets = data.dict_from_directory(
    path_data, 
    type='polars'
)

# predictions by the supervised machine learning models
preds_sml = data.dict_from_directory(
    path_predictions + '/supervised_machine_learning', 
    type='polars'
)

# predictions by the llama model
preds_llama = data.dict_from_directory(
    path_predictions + '/llama_3_1_8B', 
    type='polars'
)

Initially, the llama predictions do not contain the true labels.
We therefore concatentate the true labels and the index number from the dataset.

We further rename the prediction by the llama model from ``include`` to ``llama`` to better distinguish it from the sml predictions later.

In [3]:
import polars as pl

preds_llama = {
    subject: predictions.rename(
        {
            'include': 'llama',
            'reason': 'llama_reason'
        }
    ).with_columns(
        datasets[subject].select(
            pl.col('index').alias('index'),
            pl.col('include').alias('true_llama')
        )
    )
    for subject, predictions in preds_llama.items()
}

Verify that the dataframes have been correctly concatenated and renamed:

In [4]:
preds_llama['adhd']

llama,llama_reason,title,doi,pubmed_id,index,true_llama
bool,str,str,str,i64,i64,bool
false,""" The study population does not…","""The effectiveness of clonidine…","""https://doi.org/10.1007/bf0301…",10051933,0,false
false,""" The study population does not…","""A Controlled Trial of Sustaine…","""https://doi.org/10.1056/nejm19…",10053177,1,false
true,""" The study meets the inclusion…","""Effects of methylphenidate on …","""https://doi.org/10.1037/0021-8…",10066996,2,false
false,""" The study does not involve po…","""Spinal Clonidine Prolongs Labo…","""https://doi.org/10.1097/000005…",10072008,3,false
false,""" The study population does not…","""LowDose Clozapine for the Trea…","""https://doi.org/10.1056/nejm19…",10072410,4,false
…,…,…,…,…,…,…
false,""" The study population does not…","""Combination treatment with clo…","""https://doi.org/10.1016/s0924-…",9928923,845,false
false,""" The study population does not…","""Prolactin Levels and Adverse E…","""https://doi.org/10.1097/000047…",9934944,846,false
false,""" The study does not focus on p…","""Dexmedetomidine Failed to Bloc…","""https://doi.org/10.1097/000005…",9952147,847,false


In the next cell, we join the predictions made by all models into one DataFrame:

In [5]:
joint_test_set = {
    subject: predictions.join(
        preds_llama[subject], on='index'
    ).drop(['true_llama'])
    for subject, predictions in preds_sml.items()
}

The ```joint_test_set``` variable now contains only the articles within the sets sets.

For each article there is the ground truth ``true``, the prediction by each respective model as well as llama's reasoning:

In [6]:
joint_test_set['animal_depression'].head(5)

index,true,logistic_regression,random_forest,support_vector_machine,naive_bayes,llama,llama_reason,title,doi,pubmed_id
i64,bool,bool,bool,bool,bool,bool,str,str,str,f64
4,false,false,false,false,false,false,""" The study does not investigat…","""Glycine attenuates hepatocellu…","""https://doi.org/10.1097/000032…",1.1395604e7
6,true,true,true,true,true,true,""" The study meets the inclusion…","""Brain monoamine receptors in a…","""https://doi.org/10.1007/s00702…",1.1341483e7
10,false,false,false,false,false,false,""" The study does not investigat…","""Effects of Nitrous Oxide on My…","""https://doi.org/10.1097/000005…",2.147368e6
14,false,false,false,false,false,false,""" The study does not investigat…","""Aminoimidazolecarboxaimide rib…","""https://doi.org/10.1016/0014-2…",1.804658e6
22,false,true,true,true,true,true,""" The study meets all the inclu…","""Social instability in female r…","""https://doi.org/10.1016/j.phys…",1.5811385e7


## Helper Functions
Here we define some helper functions to increase readability when calculating the scores:

In [7]:
from typing import Tuple
import numpy as np
from sklearn.metrics import confusion_matrix

def confusion_matrices(
        y_true: np.ndarray, # list of true labels
        y_pred: np.ndarray # list of predicted labels
    ) -> Tuple[np.ndarray, np.ndarray]:
    """"
    Creates confusion matrices from lists of true and predicted labels.
    Returns a normalized and a non-normalized confusion matrix.

    Args:
        y_true (np.ndarray): List of true labels.
        y_pred (np.ndarray): List of predicted labels.

    Returns:
        Tuple[np.ndarray, np.ndarray]: A tuple containing the non-normalized and normalized confusion matrices.
    """

    matrix = confusion_matrix(
           y_true=y_true, 
           y_pred=y_pred,
           normalize=None
        )
       
    matrix_norm = confusion_matrix(
        y_true=y_true, 
        y_pred=y_pred,
        normalize='true'
    )

    return matrix, matrix_norm

In [8]:
def generate_bootstrap_samples(
        y_true: np.ndarray, # true labels
        y_pred: np.ndarray, # predicted labels
        n_resamples=1000, # number of bootstrap samples, 1000 is common
    ) -> list[dict[str, np.ndarray]]:
    """
    Generate a defined number of bootstrap samples from lists of true and predicted labels.

    Args:
        y_true (np.ndarray): True labels.
        y_pred (np.ndarray): Predicted labels.
        n_resamples (int): Number of bootstrap samples to generate. Defaults to 1000.

    Returns:
        list[dict[str, np.ndarray]]: A list of dictionaries, each containing a bootstrap sample with keys 'y_true' and 'y_pred', both as numpy arrays.

    """

    # save the bootstrapped samples here
    samples = []
    

    # create n_resamples bootstrap samples
    for _ in range(n_resamples):
        indices = np.random.choice(len(y_true), len(y_true), replace=True)
        y_true_resampled = y_true[indices]
        y_pred_resampled = y_pred[indices]

        #samples.append((y_true_resampled, y_pred_resampled))
        samples.append(
            {
                'y_true': y_true_resampled,
                'y_pred': y_pred_resampled
            }
        )

    return samples

In [9]:
from typing import Callable

def bootstrap_confidence_interval(
        bootstraps: list[dict[str, np.ndarray]], # list of bootstrapped samples
        metric: Tuple[str, Callable[[np.ndarray, np.ndarray], float]], # the metric to calculate bootstrap samples for
        round_ndigits=2, # number of digits to round the results
        confidence_level=0.95 # confidence level for the CI
) -> Tuple[float, float, float, np.ndarray]:
    """"
    Calculate the mean and confidence interval for a given metric using bootstrap samples.

    Args:
        bootstraps (list[dict[str, np.ndarray]]): List of bootstrapped samples, each containing 'y_true' and 'y_pred'.
        metric (Tuple[str, Callable[[np.ndarray, np.ndarray], float]]): A tuple containing the name of the metric and the function to calculate it.
        round_ndigits (int): Number of digits to round the results. Defaults to 2.
        confidence_level (float): Confidence level for the confidence interval. Defaults to 0.95.

    Returns:
        Tuple[float, float, float, np.ndarray]: A tuple containing the mean, lower bound, upper bound of the confidence interval, and the distribution of scores for the metric.
    """
    
    # the name of the metric and the function to calculate it
    metric_name, metric_function = metric[0], metric[1]

    # save the scores for each bootstrap sample
    scores = np.zeros(len(bootstraps), dtype=float) # pre-allocate memory for scores
    valid_count = 0 # trim scores to only valid ones afterwards

    # calculate the score for each bootstrap sample
    for _, sample in enumerate(bootstraps):

        y_true = sample['y_true']
        y_pred = sample['y_pred']

        try:
            # calculate the score
            if metric_name in ['precision', 'recall']: # pass zero_division argument for precision and recall
                score = metric_function(
                    y_true, 
                    y_pred, 
                    zero_division=0 # set to 0 to avoid errors when there are no positive predictions
                )
            else: # call the metric function without additional arguments
                score = metric_function(
                    y_true, 
                    y_pred,
                )
           
            scores[valid_count] = score  # add the score to the distribution
            valid_count += 1 # increment the count of valid scores
        except:
            continue # skip the sample if an error occurs
        
    # trim the scores to only valid ones
    scores = scores[:valid_count]
    
    # calculate the mean and lower and upper bounds
    mean = np.mean(scores)
    lower = np.percentile(scores, (1 - confidence_level) / 2 * 100) # 2.5th percentile
    upper = np.percentile(scores, (1 + confidence_level) / 2 * 100) # 97.5th percentile

    # round the results
    mean = round(mean, round_ndigits)
    lower = round(lower, round_ndigits)
    upper = round(upper, round_ndigits)

    return mean, lower, upper, scores

In [10]:
from confidenceinterval import f1_score

def delta_method_f1_score(
        y_true: np.ndarray, # true labels
        y_pred: np.ndarray, # predicted labels
        round_ndigits: int = 2, # number of digits to round the result to
        confidence_level: float = 0.95 # confidence level for the confidence interval
) -> Tuple[float, float, float]:
    """
    Calculate the f1 score and its confidence interval using the delta method.

    Args:
        y_true (np.ndarray): True labels.
        y_pred (np.ndarray): Predicted labels.
        round_ndigits (int, optional): Number of digits to round the result to. Defaults to 2.
        confidence_level (float, optional): Confidence level for the confidence interval. Defaults to 0.95.

    Returns:
        Tuple[float, float, float]: A tuple containing the f1 score, lower bound of the confidence interval, and upper bound of the confidence interval, all rounded to the specified number of digits.
    """

    # Calculate the f1 score and confidence interval using the delta method by Takahashi et al.
    result = f1_score(
        y_true,
        y_pred,
        confidence_level=confidence_level,
        average="binary",
        method="takahashi"
    )

    # Unpack the result tuple: (estimate, (lower, upper))
    estimate, (lower_bound, upper_bound) = result

    mean = round(estimate, round_ndigits)
    lower = round(lower_bound, round_ndigits)
    upper = round(upper_bound, round_ndigits)

    return mean, lower, upper

## Definitions
Some definitions prior to calculating the scores:

### Estimators

In [11]:
estimators = [
    'logistic_regression',
    'random_forest',
    'support_vector_machine',
    'naive_bayes',
    'llama'
]

### Metric Functions

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
from imblearn.metrics import specificity_score

metrics = {
    'accuracy': accuracy_score,
    'precision': precision_score,
    'recall': recall_score,
    'f1': delta_method_f1_score,
    'specificity': specificity_score,
}

### Note on F1 Score Calculation
The F1 score is calculated using the **delta method** (Takahashi's method) from the `confidenceinterval` package with `confidence_level=0.95` and `average="binary"`.

All other metrics (accuracy, precision, recall, F2, specificity) are calculated using the **bootstrap method** as before.

### Scores Table
Here we define the structure for a dataframe which will collect the mean scoes and confidence intervals which have been calculated for each combination of dataset and estimator:

In [13]:
schema_context = {
    'dataset': pl.String,
    'estimator': pl.String,
}

schema_metrics = {
    metric: pl.Struct(
        [
            pl.Field(f'{metric}_mean', pl.Float64),
            pl.Field(f'{metric}_lower', pl.Float64),
            pl.Field(f'{metric}_upper', pl.Float64),
        ]
    )
    for metric in metrics.keys()
}

schema_scores = schema_context | schema_metrics

In [14]:
schema_scores

{'dataset': String,
 'estimator': String,
 'accuracy': Struct({'accuracy_mean': Float64, 'accuracy_lower': Float64, 'accuracy_upper': Float64}),
 'precision': Struct({'precision_mean': Float64, 'precision_lower': Float64, 'precision_upper': Float64}),
 'recall': Struct({'recall_mean': Float64, 'recall_lower': Float64, 'recall_upper': Float64}),
 'f1': Struct({'f1_mean': Float64, 'f1_lower': Float64, 'f1_upper': Float64}),
 'specificity': Struct({'specificity_mean': Float64, 'specificity_lower': Float64, 'specificity_upper': Float64})}

### Results Dictionary
Next, we define the structure for a dictionary ``results`` which will contain every aspect of evaluation, the scores, bootstrap samples and calculated scores:

- The ``scores`` table will contain an overview over the mean and confidence interval for every score and every combination of dataset and estimator
- For each dataset:
    - An ``sklearn.metrics.classification_report``
    - Both a normalized and a non-normalized ``sklearn.metrics.confusion_matrix``
    - The 1000 bootstrap samples with respective scores, from which the confidence intervals have been computed

In [15]:
results = {
    'scores': pl.DataFrame([], schema=schema_scores),
    'datasets': {
        subject: {
            estimator: {
                'classification_report': None,
                'matrix': {},
                'bootstrap': {
                    'samples': None,
                    'scores': {
                        metric: None
                        for metric in metrics
                    }

                }
            }
            for estimator in estimators
        }
        for subject in joint_test_set.keys()
        
    }
}

## Calculation
Finally, we calculate confusion matrices, bootstrap samples and derived performance metrics with confidence interval for each combination of estimator and dataset, to save everything within the ``results`` dictionary:

In [16]:
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report

# iterate over the datasets
for subject, dataset in tqdm(
    iterable=joint_test_set.items(),  # iterate over the datasets
    desc='Datasets', # description for the progress bar
    total=len(joint_test_set), 
    leave=True # leave the progress bar visible after completion
):


    # save the predictions within a variable
    predictions_only = dataset.select(
        pl.exclude(
            ['index', 'true', 'llama_reason', 'title', 'doi', 'pubmed_id']
        )
    )


    # iterate over the estimators
    for estimator in tqdm(
        iterable=predictions_only.iter_columns(),
        desc='Estimators',
        total=len(predictions_only.columns),
        leave=False
    ):
        
        
        model = estimator.name  # to access the results dictionary
        y_true = dataset['true'].to_numpy() # true labels

        # predictions by the current estimator to calculate on
        y_pred = estimator.to_numpy()  

        # add the classification report
        report = classification_report(y_true, y_pred)
        results['datasets'][subject][model]['classification_report'] = report

        # add the confusion matrices
        matrix, matrix_norm = confusion_matrices(y_true, y_pred)
        results['datasets'][subject][model]['matrix']['absolute'] = matrix
        results['datasets'][subject][model]['matrix']['norm'] = matrix_norm

        # save the scores for the current estimator in here
        estimator_scores = {}

        # generate bootstrap samples for the current estimator
        bootstraps = generate_bootstrap_samples(y_true, y_pred)
        
        # add the bootstraps to the results for traceability
        results['datasets'][subject][model]['bootstrap']['samples'] = bootstraps

        # calculate each metric with confidence intervals by bootstrapping
        for metric, function in metrics.items():
            
            # calculate f1 using the delta method
            if metric == 'f1':
                # Use the delta method for F1 score
                mean, lower, upper = delta_method_f1_score(y_true, y_pred)
                bootstrap_scores = None # no bootstrap scores for the delta method
            # calculate the other metrics using bootstrapping
            else:
                mean, lower, upper, bootstrap_scores = bootstrap_confidence_interval(bootstraps, (metric, function))
                
            # Create a Series with the struct values
            struct_series = pl.Series(
                name=metric,
                values=[(mean, lower, upper)],
                dtype=pl.Struct(
                    [
                        pl.Field(f'{metric}_mean', pl.Float64),
                        pl.Field(f'{metric}_lower', pl.Float64),
                        pl.Field(f'{metric}_upper', pl.Float64),
                    ]
                )
            )

            # add the results struct to the list of scores
            estimator_scores[metric] = struct_series[0]

            # add the bootstrap scores to the results for traceability
            results['datasets'][subject][model]['bootstrap']['scores'][metric] = bootstrap_scores

        # add the results to the scores dataframe
        scores = pl.DataFrame({
            estimator: [values]
            for estimator, values in estimator_scores.items()
        })
        
        # add a dataset and estimator column to the scores dataframe
        scores = scores.with_columns(
            pl.lit(subject, pl.String).alias('dataset'),
            pl.lit(model, pl.String).alias('estimator')
        )

        # reorder the dataframe for stacking
        column_order = ['dataset', 'estimator'] + list(metrics.keys())
        scores = scores.select(column_order)

        results['scores'] = results['scores'].vstack(scores)

Datasets:   0%|          | 0/5 [00:00<?, ?it/s]

Estimators:   0%|          | 0/5 [00:00<?, ?it/s]

Estimators:   0%|          | 0/5 [00:00<?, ?it/s]

Estimators:   0%|          | 0/5 [00:00<?, ?it/s]

Estimators:   0%|          | 0/5 [00:00<?, ?it/s]

Estimators:   0%|          | 0/5 [00:00<?, ?it/s]

# Export

In [17]:
with open('./results.pkl', 'wb') as file:
    import pickle
    pickle.dump(results, file, protocol=pickle.HIGHEST_PROTOCOL)